# ML-09 â€” Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** Content refreshed more frequently ranks higher.
*Methodology Question:* Are the labels derived from historical clicks that naturally decay over time, or manual annotation? Does the correlation hold across all content domains or is it heavily skewed by news/time-sensitive topics?

**Finding 2:** Our Random Forest model predicts decline with 85% accuracy.
*Methodology Question:* Does the validation design carry this claim? Specifically, did the train/test split randomize rows, or did it properly group by client/domain to prevent structural data leakage?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load Data
df = pd.read_csv("../../../STARTERPACK ML/data/raw/content_refresh_anonymized.csv")
df = df.dropna(subset=["trend_direction"])
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
df["has_word_count"] = df["word_count"].notnull().astype(int)
df["word_count"] = df["word_count"].fillna(0)
df["avg_position"] = df["avg_position"].fillna(0)

features = ["days_since_last_update", "ctr", "impressions_90d", "clicks_90d", "avg_position", "engagement_rate", "word_count", "has_word_count"]
X = df[features]
y = df["is_declining"]
groups = df["client_id"]

# BEFORE: Naive Random Split (Causes leakage because same client rows end up in train and test)
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X, y, test_size=0.2, random_state=42)
rf_naive = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
rf_naive.fit(X_train_n, y_train_n)
acc_naive = accuracy_score(y_test_n, rf_naive.predict(X_test_n))

# AFTER: Honest Split (Grouped by Client)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, y_train_g = X.iloc[train_idx], y.iloc[train_idx]
X_test_g, y_test_g = X.iloc[test_idx], y.iloc[test_idx]

rf_group = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
rf_group.fit(X_train_g, y_train_g)
acc_group = accuracy_score(y_test_g, rf_group.predict(X_test_g))

print(f"BEFORE (Naive Split Accuracy) : {acc_naive:.2%}")
print(f"AFTER  (Honest Split Accuracy): {acc_group:.2%}")
print("Observation: The naive split overestimates performance due to client-level data leakage.")


BEFORE (Naive Split Accuracy) : 66.58%
AFTER  (Honest Split Accuracy): 57.65%
Observation: The naive split overestimates performance due to client-level data leakage.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Audit: Correlation check for target leakage
corr = df[features + ["is_declining"]].corr()["is_declining"].sort_values(ascending=False)
print("Feature Correlation with Target (Absolute Leakage Check):")
print(corr)
print("\nConclusion: No feature has a suspiciously near-perfect correlation (e.g., > 0.9) that would indicate trivial target leakage.")


Feature Correlation with Target (Absolute Leakage Check):
is_declining              1.000000
word_count                0.118863
has_word_count            0.090431
days_since_last_update    0.081383
engagement_rate          -0.012743
impressions_90d          -0.018175
avg_position             -0.029035
clicks_90d               -0.039680
ctr                      -0.061911
Name: is_declining, dtype: float64

Conclusion: No feature has a suspiciously near-perfect correlation (e.g., > 0.9) that would indicate trivial target leakage.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Bold Claim:** Our machine learning model accurately predicts exactly which content will decline so you can update it immediately.

**Rewritten Safe Claim:** Our model provides directional decision-support by measuring content decay indicators. It observes historical trends to help prioritize which articles might benefit from a refresh.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled â€” markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime â†’ Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` â€” then submit your repo URL on the card. Done.